STEP 1: Environment Setup, PyTorch, and CUDA

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\n✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ CUDA version: {torch.version.cuda}")

Mon Dec  8 06:40:49 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

STEP 2: Clone the Github Repository

In [ ]:
# If you re-run, you might want to remove the old folder first
!rm -rf MC-Net
!git clone https://github.com/ycwu1997/MC-Net.git
%cd MC-Net
!ls

Cloning into 'MC-Net'...
remote: Enumerating objects: 228, done.
remote: Counting objects: 100% (228/228), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 228 (delta 90), reused 217 (delta 85), pack-reused 0 (from 0)
Receiving objects: 100% (228/228), 238.09 MiB | 16.85 MiB/s, done.
Resolving deltas: 100% (90/90), done.
Updating files: 100% (46/46), done.
/content/MC-Net
code  LICENSE  pretrained_pth  train_mcnet_2d.sh
data  model    README.md       train_mcnet_3d.sh


STEP 3: Install other dependencies needed for this framework

In [ ]:
import sys
import subprocess

# Use the same Python that the notebook is using
pip_cmd = [sys.executable, "-m", "pip", "install", "-q"]

# 1) Make sure numpy is a version that plays nicely with MedPy (<2.0)
subprocess.run(pip_cmd + ["numpy<2.0,>=1.24"], check=True)

# 2) Install the other dependencies
subprocess.run(pip_cmd + [
    "scikit-image",
    "scikit-learn",
    "scipy",
    "SimpleITK",
    "tensorboardX",
    "nibabel",
    "tqdm",
    "medpy",
    "h5py",
    "opencv-python",

], check=True)

print("Done installing dependencies (except PyTorch/CUDA).")

Done installing dependencies (except PyTorch/CUDA).


In [ ]:
import numpy as np
import skimage
import sklearn
import scipy
import SimpleITK
import numpy as np
import tensorboardX
import nibabel as nib
import tqdm
import medpy
import h5py

print("All imports OK")
print("numpy:", np.__version__)

STEP 4: Download DataSet for LA

In [ ]:
#make sure you move to the right file path
!git clone https://github.com/yulequan/UA-MT.git
%cd UA-MT/data
!ls

Cloning into 'UA-MT'...
remote: Enumerating objects: 257, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 257 (delta 6), reused 5 (delta 5), pack-reused 242 (from 1)
Receiving objects: 100% (257/257), 315.19 MiB | 13.87 MiB/s, done.
Resolving deltas: 100% (13/13), done.
Updating files: 100% (120/120), done.
/content/MC-Net/UA-MT/data
'2018LA_Seg_Training Set'   test.list   train.list


In [ ]:
!grep -R "np.int" -n /content/MC-Net

/content/MC-Net/UA-MT/code/dataloaders/utils.py:65:    label_mask = np.zeros((mask.shape[0], mask.shape[1]), dtype=np.int16)
/content/MC-Net/code/utils/test_patch.py:161:    label_map = (score_map[0]>0.5).astype(np.int32)
/content/MC-Net/code/utils/test_patch.py:227:    label_map = (score_map[0]>0.5).astype(np.int32)


STEP 5: Start training for LA

In [ ]:
%cd /content/MC-Net
# e.g., for 10% labels on LA
!python ./code/train_mcnet_3d.py --dataset_name LA --model mcnet3d_v2 --labelnum 8 --gpu 0 --temperature 0.1

Streaming output truncated to the last 5000 lines.
iteration 10532 : loss : 0.057673, loss_d: 0.087104, loss_cosist: 0.014121
 70%|█████████████████▌       | 2633/3751 [6:35:15<2:31:46,  8.15s/it]iteration 10533 : loss : 0.136006, loss_d: 0.203038, loss_cosist: 0.034487
iteration 10534 : loss : 0.061205, loss_d: 0.087316, loss_cosist: 0.017547
iteration 10535 : loss : 0.050162, loss_d: 0.077904, loss_cosist: 0.011210
iteration 10536 : loss : 0.054438, loss_d: 0.077729, loss_cosist: 0.015574
 70%|█████████████████▌       | 2634/3751 [6:35:23<2:30:13,  8.07s/it]iteration 10537 : loss : 0.062635, loss_d: 0.095044, loss_cosist: 0.015113
iteration 10538 : loss : 0.044839, loss_d: 0.069936, loss_cosist: 0.009871
iteration 10539 : loss : 0.055483, loss_d: 0.088559, loss_cosist: 0.011203
iteration 10540 : loss : 0.070367, loss_d: 0.077191, loss_cosist: 0.031772
 70%|█████████████████▌       | 2635/3751 [6:35:31<2:30:36,  8.10s/it]iteration 10541 : loss : 0.049903, loss_d: 0.074158, loss_cosist

In [ ]:
%cd /content/MC-Net

!python ./code/test_3d.py --dataset_name LA --model mcnet3d_v2 --exp MCNet --labelnum 16 --gpu 0

/content/MC-Net
./model/LA_MCNet_16_labeled/mcnet3d_v2_predictions/
init weight from ./model/LA_MCNet_16_labeled/mcnet3d_v2/mcnet3d_v2_best_model.pth
00,	0.89648, 0.81238, 5.38516, 1.77589
01,	0.91052, 0.83573, 5.83095, 1.26400
02,	0.90754, 0.83073, 8.06226, 1.04282
03,	0.92302, 0.85705, 4.58258, 1.46614
04,	0.90986, 0.83463, 5.38516, 1.83484
05,	0.86347, 0.75974, 10.24695, 2.29933
06,	0.93615, 0.87997, 3.00000, 1.17289
07,	0.85772, 0.75088, 9.00000, 1.85989
08,	0.93949, 0.88588, 4.12311, 1.66718
09,	0.93667, 0.88089, 3.46410, 1.02952
10,	0.86505, 0.76220, 8.36660, 2.58773
11,	0.90162, 0.82086, 4.58258, 1.20384
12,	0.91085, 0.83630, 7.81025, 1.02215
13,	0.92799, 0.86566, 4.58258, 1.04206
14,	0.92380, 0.85839, 5.09902, 1.11764
15,	0.91607, 0.84514, 6.40312, 2.63768
16,	0.85702, 0.74981, 10.48809, 2.80138
17,	0.89708, 0.81336, 5.09902, 4.19734
18,	0.94771, 0.90062, 4.12311, 1.78182
19,	0.90563, 0.82753, 7.28011, 1.99330
average metric is decoder 1 [0.906687   0.83038741 6.14573708 1.7898

STEP 6: Start training for ACDC

In [ ]:
#code to unzip the ACDC.zip
!unzip -q /content/MC-Net/data/ACDC/ACDC.zip -d /content/MC-Net/data/ACDC/

In [ ]:
#this code will train on the ACDC dataset with ~20% labeled data, completing 30,000 iteration
%cd /content/MC-Net

!python ./code/train_mcnet_2d.py --model mcnet2d_v2 --exp MCNet --labelnum 7 --gpu 0 --temperature 0.1

Streaming output truncated to the last 5000 lines.
iteration 25027 : loss : 1.524718, loss_d: 1.524240, loss_cosist: 0.005526
iteration 25028 : loss : 1.524558, loss_d: 1.524032, loss_cosist: 0.006080
iteration 25029 : loss : 1.525084, loss_d: 1.524564, loss_cosist: 0.006008
iteration 25030 : loss : 1.524691, loss_d: 1.524240, loss_cosist: 0.005202
iteration 25031 : loss : 1.531505, loss_d: 1.531064, loss_cosist: 0.005088
iteration 25032 : loss : 1.524471, loss_d: 1.524044, loss_cosist: 0.004929
iteration 25033 : loss : 1.527650, loss_d: 1.527186, loss_cosist: 0.005360
iteration 25034 : loss : 1.527037, loss_d: 1.526535, loss_cosist: 0.005799
iteration 25035 : loss : 1.526439, loss_d: 1.525964, loss_cosist: 0.005479
iteration 25036 : loss : 1.524922, loss_d: 1.524540, loss_cosist: 0.004414
 83%|████████████████████    | 2276/2728 [10:27:30<2:10:31, 17.33s/it]iteration 25037 : loss : 1.528789, loss_d: 1.528326, loss_cosist: 0.005360
iteration 25038 : loss : 1.527084, loss_d: 1.526622, l